# PromEvo on Colab (one Snakemake run)
Run the cells **top to bottom, one at a time**. Before you start, put your genome folder(s) in `My Drive/promevo_run`
(use a *small* set first). The last cell does everything: motif databases, then **Stage 01** (it asks you its questions
in input boxes), then stages 2-17.

## 1. Connect Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/promevo_run'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('Work folder:', WORK)
print(os.listdir(WORK))

## 2. Get the PromEvo code and micromamba

In [ ]:
%%bash
cd /content && rm -rf PromEvo && git clone --depth 1 https://github.com/ausswal/PromEvo.git
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
ls /content/PromEvo /content/bin

## 3. Install the tools (10-15 min, only needed once per session)

In [ ]:
%%bash
rm -rf /content/promevo_env
/content/bin/micromamba create -y -p /content/promevo_env -c conda-forge -c bioconda \
  python=3.12 pandas numpy scipy matplotlib-base seaborn biopython openpyxl scikit-learn requests \
  ncbi-datasets-cli mafft meme orthofinder diamond mcl fasttree snakemake-minimal 2>&1 | tail -8
ls /content/promevo_env/bin | grep -E "^(meme|tomtom|mafft|orthofinder|diamond|mcl|datasets|fasttree|snakemake)$"

## 4. Environment variables (re-run in every new session)

In [ ]:
import os
ENV = '/content/promevo_env'
REPO = '/content/PromEvo'
os.environ['PATH'] = ENV + '/bin:' + os.environ['PATH']
os.environ['CONDA_PREFIX'] = ENV
os.environ['MPLBACKEND'] = 'Agg'      # Colab's notebook-only plot backend does not exist in the tools env
os.environ['COMPARATIVE_PIPELINE_WORKDIR']   = WORK
os.environ['COMPARATIVE_PIPELINE_SCRIPTS']   = REPO + '/scripts'
os.environ['COMPARATIVE_PIPELINE_DATABASES'] = WORK + '/databases'
os.chdir(WORK)
print('ok')

## 5. Quick check: libraries and tools

In [ ]:
!/content/promevo_env/bin/python -c "import numpy, pandas, scipy, matplotlib, seaborn, Bio, openpyxl, sklearn, requests; print('python packages OK')"
!for t in datasets mafft meme tomtom ceqlogo orthofinder diamond mcl FastTree snakemake; do command -v $t >/dev/null && echo "OK      $t" || echo "MISSING $t"; done

## 6. Helper (run once). Lets Snakemake's Stage 01 ask you questions through input boxes

In [ ]:
# Helper: runs a command that asks questions and relays each question to a notebook input box.
import os, sys, select, subprocess, time

def run_interactive(args, idle=0.8):
    """Run a script that uses input() in its own Python, relaying prompts to notebook input boxes."""
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, bufsize=0, env=env)
    try:
        my_stdin = os.readlink(f"/proc/{p.pid}/fd/0")      # e.g. pipe:[12345] = the pipe WE write answers into
    except Exception:
        my_stdin = None
    fd, last, quiet_since = p.stdout.fileno(), b"\n", None
    try:
        open(f"/proc/{p.pid}/syscall").read(); proc_ok = True
    except Exception:
        proc_ok = False       # fallback: treat >4 s of silence as a possible question (type #wait to skip)

    def descendants(root):
        kids, out = {}, [root]
        for d in os.listdir("/proc"):
            if d.isdigit():
                try:
                    st = open(f"/proc/{d}/stat").read()
                    ppid = int(st[st.rindex(")") + 2:].split()[1])
                    kids.setdefault(ppid, []).append(int(d))
                except Exception:
                    pass
        i = 0
        while i < len(out):
            out += kids.get(out[i], []); i += 1
        return out

    def waiting_on_stdin():
        for pid in descendants(p.pid):
            try:
                if my_stdin and os.readlink(f"/proc/{pid}/fd/0") != my_stdin:
                    continue                                   # e.g. tee reading python's output
                parts = open(f"/proc/{pid}/syscall").read().split()
                if parts[0] == "0" and int(parts[1], 16) == 0:     # read(fd=0, ...)
                    return True
            except Exception:
                pass
        return False

    while True:
        r, _, _ = select.select([fd], [], [], 0.25)
        if r:
            d = os.read(fd, 65536)
            if not d:
                break
            sys.stdout.write(d.decode(errors="replace")); sys.stdout.flush()
            last, quiet_since = d[-1:], None
            continue
        if p.poll() is not None:
            continue                                   # drain remaining output, then EOF ends loop
        quiet_since = quiet_since or time.time()
        asking = waiting_on_stdin() if proc_ok else (time.time() - quiet_since > 4)
        if asking and proc_ok:
            time.sleep(0.3)
            asking = waiting_on_stdin() and not select.select([fd], [], [], 0)[0]
        if asking or (time.time() - quiet_since > idle and last != b"\n" and not proc_ok):
            ans = input()                              # notebook input box
            if ans.strip() == "#wait":
                quiet_since = time.time(); continue
            p.stdin.write((ans + "\n").encode()); p.stdin.flush()
            last, quiet_since = b"\n", None
    return p.wait()


## 7. Run everything
Databases download first, then **Stage 01 asks its questions**. Each question appears above an input box: type the answer and press Enter.
If a box appears while it is only working, type `#wait` and press Enter. Keep this tab open; it can take hours on Colab's 2 cores.

In [ ]:
run_interactive(["/content/promevo_env/bin/snakemake", "-s", "/content/PromEvo/workflow/Snakefile", "--cores", "2"])

## If it stops or the session disconnects
Reconnect and re-run cells 1, 2 (clone only), 4, 6 and 7. Finished steps are skipped, so it carries on.
To redo Stage 01 and everything after it, run the next cell once, then cell 7.

In [ ]:
%%bash
cd /content/drive/MyDrive/promevo_run
rm -f outputs/.done/1.done outputs/.done/{2..17}.done
rm -rf .snakemake/locks
echo reset

## Results
Everything is in `My Drive/promevo_run/outputs/`. Open `promoter_comparative_dashboard.html` there.